# Résultats unifiés A1-A7 — France vs Monde (une seule pipeline)

**Reproductible et hors-ligne** : ce notebook ne fetch rien. Il lit l'archive gelée de `run_unified_audit.py` (`../experiments/unified_audit/`), où chaque flux `station_information` est figé en Parquet avec horodatage UTC et SHA-256, puis audité par **la même** fonction `gbfs_toolkit.audit_static` que la France.

Deux variantes scientifiquement distinctes du **même** audit sur les **mêmes** flux archivés :
- **feed_only** : ce que GBFS seul permet (A1 non intrinsèque au flux).
- **operator** : + classification carsharing par nom d'opérateur (l'étape S1 du catalogue, jeu de mots-clés figé), qui rend A1 comparable.

Carte de repro : `seed=42`, params et version toolkit dans `RESULTS.json`, hashes par flux dans `MANIFEST.json`.

In [1]:
import json, pandas as pd
OUT = '../experiments/unified_audit'
R = json.load(open(f'{OUT}/RESULTS.json'))
man = pd.read_csv(f'{OUT}/world_fetch_manifest.csv')
verdict = pd.read_parquet(f'{OUT}/unified_verdict.parquet')
FLAGS = ['A1','A2','A3','A4','A5','A6','A7']
print('toolkit', R['toolkit_version'], '| seed', R['seed'], '| généré', R['generated_at'][:19])
print('params', R['params'])
print('corpus dans le verdict:', sorted(verdict.corpus.unique()))

toolkit 1.4.0 | seed 42 | généré 2026-06-28T08:48:46
params {'a4_sigma': 3.0, 'a7_scope': 'all', 'n_min': 20, 'bootstrap_n': 10000}
corpus dans le verdict: ['FR_catalogue', 'World_feed_only', 'World_operator']


## 1. Couverture (funnel reproductible)

Inventaire MobilityData figé, re-fetch live daté ; les flux morts sont tracés, pas masqués.

In [2]:
import collections
st = collections.Counter(s.split(':')[0] for s in man.status)
funnel = pd.Series({
    'inventaire (figé)': R['inventory_systems'],
    'publient station_information (ok)': int((man.status=='ok').sum()),
    'injoignables': st.get('unreachable',0),
    'vides': st.get('empty',0),
    'sans station_information': st.get('no_station_information',0),
})
print('stations archivées :', int(man.n_stations.sum()), '| pays ok :', man[man.status=='ok'].country_code.nunique())
funnel

stations archivées : 228294 | pays ok : 46


inventaire (figé)                    1509
publient station_information (ok)     936
injoignables                           83
vides                                 177
sans station_information              313
dtype: int64

## 2. Table unifiée comparable (variante operator = méthode du catalogue)

Une seule pipeline, `audit_static` identique, FR ⊂ World. Comptes système (flaggé ssi ≥1 station flaggée).

In [3]:
def counts(variant, corp):
    b = R[variant][corp]
    return {k: b[k]['systems_flagged'] for k in FLAGS}, b['_n_systems']
rows = {}
for corp in ['FR','non_FR','World']:
    c, n = counts('world_operator', corp)
    rows[f'{corp} (n={n})'] = c
pd.DataFrame(rows)

,FR (n=145),non_FR (n=791),World (n=936)
A1,15,21,36
A2,3,5,8
A3,65,634,699
A4,89,426,515
A5,6,24,30
A6,0,0,0
A7,29,259,288


In [4]:
# Taux (%) + IC95 bootstrap (seed 42), World, variante operator
wb = R['world_operator']['World']; nW = wb['_n_systems']
pd.DataFrame({
    'systems': {k: wb[k]['systems_flagged'] for k in FLAGS},
    'rate_%': {k: round(100*wb[k]['systems_flagged']/nW,1) for k in FLAGS},
    'CI95_%': {k: wb[k]['ci95_rate_pct'] for k in FLAGS},
})

,systems,rate_%,CI95_%
A1,36,3.8,"[2.67, 5.13]"
A2,8,0.9,"[0.32, 1.5]"
A3,699,74.7,"[71.9, 77.46]"
A4,515,55.0,"[51.82, 58.23]"
A5,30,3.2,"[2.14, 4.38]"
A6,0,0.0,"[0.0, 0.0]"
A7,288,30.8,"[27.88, 33.65]"


## 3. Finding : A1 n'est pas intrinsèque au flux

Sur les **mêmes** flux archivés, A1 passe de **0** (feed_only) à **36** (operator). Un flux GBFS ne déclare pas « carsharing » : A1 dépend d'une métadonnée opérateur, hors du flux. A2-A5 et A7 sont, elles, intrinsèques (inchangées entre les deux variantes, hormis le léger transfert A3->A1).

In [5]:
comp = pd.DataFrame({
    'World feed_only': {k: R['world_feed_only']['World'][k]['systems_flagged'] for k in FLAGS},
    'World operator':  {k: R['world_operator']['World'][k]['systems_flagged'] for k in FLAGS},
    'FR parquet post-audit': R['france_catalogue_counts'],
})
comp

,World feed_only,World operator,FR parquet post-audit
A1,0,36,17
A2,8,8,1
A3,715,699,41
A4,515,515,88
A5,30,30,4
A6,0,0,0
A7,288,288,32


Lecture : A2-A5/A7 sont stables (intrinsèques au flux) ; seul A1 (et le léger report A3->A1) bouge avec la métadonnée opérateur. Pour le papier, c'est la frontière nette entre « ce que GBFS seul certifie » et « ce qui exige une couche opérateur ».

## 4. Figure : taux mondiaux par classe (variante operator), IC95

In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
ks = ['A1','A2','A3','A4','A5','A7']
vals = [wb[k]['systems_flagged']/nW*100 for k in ks]
los = [vals[i]-wb[k]['ci95_rate_pct'][0] for i,k in enumerate(ks)]
his = [wb[k]['ci95_rate_pct'][1]-vals[i] for i,k in enumerate(ks)]
fig, ax = plt.subplots(figsize=(6,3))
ax.bar(ks, vals, yerr=[los,his], capsize=4, color='#264653')
ax.set_ylabel('% systèmes flaggés (World)')
ax.set_title('Audit unifié mondial (n=%d), variante operator, IC95 bootstrap' % nW)
plt.tight_layout(); plt.show()

/tmp/ipykernel_34699/3702620092.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 5. Reproductibilité

```
python run_unified_audit.py --audit-only   # audit déterministe depuis l'archive (aucun fetch)
python run_unified_audit.py                # re-fetch complet, résumable, regèle l'archive
```
Verdicts byte-identiques à chaque exécution ; IC reproductibles via `seed=42` ; provenance et intégrité dans `MANIFEST.json`.